# Exercise - Pricing Swaptions


#### Notation Commands

$$\newcommand{\Black}{\mathcal{B}}
\newcommand{\Blackcall}{\Black_{\mathrm{call}}}
\newcommand{\Blackput}{\Black_{\mathrm{put}}}
\newcommand{\EcondS}{\hat{S}_{\mathrm{conditional}}}
\newcommand{\Efwd}{\mathbb{E}^{T}}
\newcommand{\Ern}{\mathbb{E}^{\mathbb{Q}}}
\newcommand{\Tfwd}{T_{\mathrm{fwd}}}
\newcommand{\Tunder}{T_{\mathrm{bond}}}
\newcommand{\accint}{A}
\newcommand{\carry}{\widetilde{\cpn}}
\newcommand{\cashflow}{C}
\newcommand{\convert}{\phi}
\newcommand{\cpn}{c}
\newcommand{\ctd}{\mathrm{CTD}}
\newcommand{\disc}{Z}
\newcommand{\done}{d_{1}}
\newcommand{\dt}{\Delta t}
\newcommand{\dtwo}{d_{2}}
\newcommand{\flatvol}{\sigma_{\mathrm{flat}}}
\newcommand{\flatvolT}{\sigma_{\mathrm{flat},T}}
\newcommand{\float}{\mathrm{flt}}
\newcommand{\freq}{m}
\newcommand{\futprice}{\mathcal{F}(t,T)}
\newcommand{\futpriceDT}{\mathcal{F}(t+h,T)}
\newcommand{\futpriceT}{\mathcal{F}(T,T)}
\newcommand{\futrate}{\mathscr{f}}
\newcommand{\fwdprice}{F(t,T)}
\newcommand{\fwdpriceDT}{F(t+h,T)}
\newcommand{\fwdpriceT}{F(T,T)}
\newcommand{\fwdrate}{f}
\newcommand{\fwdvol}{\sigma_{\mathrm{fwd}}}
\newcommand{\fwdvolTi}{\sigma_{\mathrm{fwd},T_i}}
\newcommand{\grossbasis}{B}
\newcommand{\hedge}{\Delta}
\newcommand{\ivol}{\sigma_{\mathrm{imp}}}
\newcommand{\logprice}{p}
\newcommand{\logyield}{y}
\newcommand{\mat}{(n)}
\newcommand{\nargcond}{d_{1}}
\newcommand{\nargexer}{d_{2}}
\newcommand{\netbasis}{\tilde{\grossbasis}}
\newcommand{\normcdf}{\mathcal{N}}
\newcommand{\notional}{K}
\newcommand{\pfwd}{P_{\mathrm{fwd}}}
\newcommand{\pnl}{\Pi}
\newcommand{\price}{P}
\newcommand{\probexer}{\hat{\mathcal{P}}_{\mathrm{exercise}}}
\newcommand{\pvstrike}{K^*}
\newcommand{\refrate}{r^{\mathrm{ref}}}
\newcommand{\rrepo}{r^{\mathrm{repo}}}
\newcommand{\spotrate}{r}
\newcommand{\spread}{s}
\newcommand{\strike}{K}
\newcommand{\swap}{\mathrm{sw}}
\newcommand{\swaprate}{\cpn_{\swap}}
\newcommand{\tbond}{\mathrm{fix}}
\newcommand{\ttm}{\tau}
\newcommand{\value}{V}
\newcommand{\vega}{\nu}
\newcommand{\years}{\tau}
\newcommand{\yearsACT}{\tau_{\mathrm{act/360}}}
\newcommand{\yield}{Y}$$


# 1. Pricing the Swaption


## Swaption Vol Data

The file `data/swaption_vol_data_2025-06-30.xlsx` has market data on the implied volatility skews for swaptions. Note that it has several columns:
* `expry`: expiration of the swaption
* `tenor`: tenor of the underlying swap
* `model`: the model by which the volatility is quoted. (All are Black.)
* `-200`, `-100`, etc.: The strike listed as difference from ATM strike (bps). Note that ATM is considered to be the **forward swap rate** which you can calculate.


Your data: you will use a single row of this data for the `1x4` swaption.
* date: `2025-06-30`
* expiration: 1yr
* tenor: 4yrs


## Rate Data

The file `data/cap_curves_2025-06-30.xlsx` gives 
* SOFR swap rates, 
* their associated discount factors
* their associated forward interest rates.

You will not need the cap data (flat or forward vols) for this problem.


## The Swaption

Consider the following swaption with the following features:
* underlying is a fixed-for-floating (SOFR) swap
* the underlying swap has **quarterly** payment frequency
* this is a **payer** swaption, which gives the holder the option to **pay** the fixed swap rate and receive SOFR.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
VOL_FILE_PATH = '../data/swaption_vol_data_2025-06-30.xlsx'
RATES_FILE_PATH = '../data/cap_curves_2025-06-30.xlsx'
vol_data = pd.read_excel(VOL_FILE_PATH, sheet_name='bloomberg vcub')
rates_data = pd.read_excel(RATES_FILE_PATH, sheet_name='rate curves 2025-06-30')

### 1.1
Calculate the (relevant) forward swap rate. That is, the one-year forward 4-year swap rate.



In [2]:
def calc_spot_swap_rate(rates_df, fraction=0.25):
    """Returns a copy of rates_df with spot_swap_rate column added."""
    rates_df = rates_df.copy()   # FIX: don't mutate the caller's DataFrame
    rates_df['sum_discounts'] = rates_df['discounts'].cumsum()
    rates_df['spot_swap_rate'] = (1 - rates_df['discounts']) / (fraction * rates_df['sum_discounts'])
    return rates_df

def calc_period_forward_rates(rates_df, fraction=0.25):
    """
    Computes the simple (money-market) forward rate for each period.
    Convention: f(t-frac, t) = (Z(t-frac)/Z(t) - 1) / frac
    This matches the 'forwards' column in the rates data.
    NOTE: renamed from calc_forward_swap_rate to avoid collision with the
    forward-swap-rate function defined in section 1.1.
    """
    # FIX: was np.pow(Z(t-frac)/Z(t), -1/frac) - 1  (annually-compounded rate)
    #      correct formula for simple rate:
    rates_df = rates_df.copy()
    rates_df['forward_discount_rate'] = rates_df['discounts'].shift(1) / rates_df['discounts']
    rates_df['forward_rate'] = (rates_df['forward_discount_rate'] - 1) / fraction
    return rates_df

In [3]:
rates_df2 = calc_spot_swap_rate(rates_data.copy())
display(rates_df2.head(20))

rates_df3 = calc_period_forward_rates(rates_data.copy())
display(rates_df3.head(20))

,tenor,swap rates,spot rates,discounts,forwards,flat vols,fwd vols,sum_discounts,spot_swap_rate
0,0.25,0.042353,0.042353,0.989523,NaN,NaN,NaN,0.989523,0.042353
1,0.50,0.040859,0.040852,0.979883,0.039351,0.156842,0.156842,1.969405,0.040859
2,0.75,0.039391,0.039372,0.971043,0.036414,0.180709,0.201708,2.940448,0.039391
3,1.00,0.038115,0.038083,0.962807,0.034217,0.204576,0.240464,3.903255,0.038115
4,1.25,0.036704,0.036653,0.955417,0.030938,0.242127,0.328341,4.858672,0.036704
5,1.50,0.035655,0.035590,0.948239,0.030280,0.268642,0.336521,5.806911,0.035655
6,1.75,0.034942,0.034868,0.941054,0.030542,0.285885,0.336809,6.747965,0.034942
7,2.00,0.034453,0.034374,0.933835,0.030919,0.295615,0.328654,7.681800,0.034453
8,2.25,0.034000,0.033916,0.926827,0.030248,0.299596,0.312413,8.608627,0.034000
9,2.50,0.033750,0.033665,0.919605,0.031414,0.299589,0.296022,9.528232,0.033750


,tenor,swap rates,spot rates,discounts,forwards,flat vols,fwd vols,forward_discount_rate,forward_rate
0,0.25,0.042353,0.042353,0.989523,NaN,NaN,NaN,NaN,NaN
1,0.50,0.040859,0.040852,0.979883,0.039351,0.156842,0.156842,1.009838,0.039351
2,0.75,0.039391,0.039372,0.971043,0.036414,0.180709,0.201708,1.009104,0.036414
3,1.00,0.038115,0.038083,0.962807,0.034217,0.204576,0.240464,1.008554,0.034217
4,1.25,0.036704,0.036653,0.955417,0.030938,0.242127,0.328341,1.007734,0.030938
5,1.50,0.035655,0.035590,0.948239,0.030280,0.268642,0.336521,1.007570,0.030280
6,1.75,0.034942,0.034868,0.941054,0.030542,0.285885,0.336809,1.007636,0.030542
7,2.00,0.034453,0.034374,0.933835,0.030919,0.295615,0.328654,1.007730,0.030919
8,2.25,0.034000,0.033916,0.926827,0.030248,0.299596,0.312413,1.007562,0.030248
9,2.50,0.033750,0.033665,0.919605,0.031414,0.299589,0.296022,1.007853,0.031414


The `forward_rate` column now matches the `forwards` column in the data (simple/money-market convention).

In [4]:
# Compute the forward swap rate S(t1, t1+T) for any T > 0
# Formula: S(t1, T) = (Z(t1) - Z(t1+T)) / (frac * sum_{t1 < t <= t1+T} Z(t))
forward_rates_df = rates_data[['tenor', 'forwards', 'discounts']].copy()

def calc_forward_swap_rate(rates_df, t1, fraction=0.25):
    """
    Returns a DataFrame (tenors > t1) with 'forward_swap_rate' giving
    the t1-forward swap rate for each horizon.
    """
    mask = rates_df['tenor'] > t1
    rates = rates_df[mask].copy()   # FIX: .copy() avoids SettingWithCopyWarning
                                    #      and is safe under pandas CoW (>= 2.0)
    discount0 = rates_df.loc[rates_df['tenor'] == t1, 'discounts'].values[0]
    rates['sum_discounts'] = rates['discounts'].cumsum()
    rates['forward_swap_rate'] = (discount0 - rates['discounts']) / (fraction * rates['sum_discounts'])
    return rates

In [5]:
t1 = 1; t2 = 4
forward_swap_rate_df = calc_forward_swap_rate(forward_rates_df, t1)
display(forward_swap_rate_df.head(20))
fw_sw_rate = forward_swap_rate_df.loc[forward_swap_rate_df['tenor'] == t1 + t2, 'forward_swap_rate'].values[0]
print(f'forward swap rate for 1y 4y: {fw_sw_rate}')


,tenor,forwards,discounts,sum_discounts,forward_swap_rate
4,1.25,0.030938,0.955417,0.955417,0.030938
5,1.50,0.030280,0.948239,1.903656,0.030610
6,1.75,0.030542,0.941054,2.844710,0.030588
7,2.00,0.030919,0.933835,3.778545,0.030669
8,2.25,0.030248,0.926827,4.705372,0.030586
9,2.50,0.031414,0.919605,5.624977,0.030722
10,2.75,0.032242,0.912252,6.537228,0.030934
11,3.00,0.032987,0.904790,7.442018,0.031183
12,3.25,0.032816,0.897427,8.339445,0.031359
13,3.50,0.032700,0.890150,9.229596,0.031488


forward swap rate for 1y 4y: 0.03269770231881184


### 1.2
Price the swaptions at the quoted implied volatilites and corresponding strikes, all using the just-calculated forward swap rate as the underlying.



In [6]:
sw_1y_4y_vols = vol_data.loc[(vol_data['expiration'] == 1) & (vol_data['tenor'] == 4), :]
display(sw_1y_4y_vols)
atm_rate = fw_sw_rate
from scipy.stats import norm
def calc_swaption_price(strike, fw_sw_rate, vol, forward_rates_df, fraction = 0.25, expiry = 1, tenor = 4, swaption_type = 'payer'):
    ''' returns the price of a payer/receiver swaption'''
    if vol > 1:
        vol = vol / 100
    required_discounts = forward_rates_df.loc[(forward_rates_df['tenor'] > expiry) & (forward_rates_df['tenor'] <= (expiry + tenor)), 'discounts' ].values
    discount_factor_sw = fraction * (required_discounts.sum())
    d1 = (np.log(fw_sw_rate/strike) + vol**2 / 2 * expiry) / (vol * np.sqrt(expiry))
    d2 = d1 - (vol * np.sqrt(expiry))
    price = 0
    print(f'discount_factor_sw: {discount_factor_sw}, d1: {d1}, d2: {d2}')
    if swaption_type == 'payer':
        price = discount_factor_sw * (fw_sw_rate * norm.cdf(d1) - strike * norm.cdf(d2))
    else:
        price = discount_factor_sw * (strike * norm.cdf(-d2) - fw_sw_rate * norm.cdf(-d1))
    return price



,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885


In [7]:
sw_1y_4y_vols.columns


Index([ 'reference', 'instrument',      'model',       'date', 'expiration',
            'tenor',         -200,         -100,          -50,          -25,
                  0,           25,           50,          100,          200],
      dtype='object')

In [8]:
fw_sw_rate

np.float64(0.03269770231881184)

In [9]:
strike_diffs = [-200, -100, -50, -25, 0, 25, 50, 100, 200]
strikes, prices = [], []
atm_price = None
notional = 100
fraction = 0.25
for diff in strike_diffs:
    strike = fw_sw_rate + diff/10000
    vol = sw_1y_4y_vols[diff].values[0]
    
    # print(f'calculating price for strike: {strike}, vol: {vol}')
    price = calc_swaption_price(strike, fw_sw_rate, vol, forward_rates_df, fraction = fraction, swaption_type = 'payer') * notional
    strikes.append(strike)
    prices.append(price)
    print(f"Strike: {strike}, Price: {price}")
    if strike == fw_sw_rate:
        atm_price = price
    
swaption_price_df = pd.DataFrame({'strike': strikes, 'price': prices})
display(swaption_price_df)

discount_factor_sw: 3.603237875869894, d1: 2.0106221007016267, d2: 1.4665721007016268
Strike: 0.01269770231881184, Price: 7.271096405156423
discount_factor_sw: 3.603237875869894, d1: 1.1393856379514864, d2: 0.7537356379514863
Strike: 0.022697702318811838, Price: 3.9480488326097305
discount_factor_sw: 3.603237875869894, d1: 0.6588080478444112, d2: 0.31955804784441116
Strike: 0.02769770231881184, Price: 2.5362346966240406
discount_factor_sw: 3.603237875869894, d1: 0.4080287991204582, d2: 0.08607879912045818
Strike: 0.03019770231881184, Price: 1.9431299983348316
discount_factor_sw: 3.603237875869894, d1: 0.15414999999999998, d2: -0.15414999999999998
Strike: 0.03269770231881184, Price: 1.443366149589084
discount_factor_sw: 3.603237875869894, d1: -0.09816840877395136, d2: -0.3962184087739513
Strike: 0.03519770231881184, Price: 1.0423913720161138
discount_factor_sw: 3.603237875869894, d1: -0.3435930054694185, d2: -0.6345430054694186
Strike: 0.03769770231881184, Price: 0.7365597575253519
disc

,strike,price
0,0.012698,7.271096
1,0.022698,3.948049
2,0.027698,2.536235
3,0.030198,1.943130
4,0.032698,1.443366
5,0.035198,1.042391
6,0.037698,0.736560
7,0.042698,0.355738
8,0.052698,0.087983


### 1.3
To consider how the expiration and tenor matter, calculate the prices of a few other swaptions for comparison. 
* No need to get other implied vol quotes--just use the ATM implied vol you have for the swaption above. (Here we are just interested in how Black's formula changes with changes in tenor and expiration.)
* No need to calculate for all the strikes--just do the ATM strike.

Alternate swaptions
* The 3mo x 4yr swaption
* The 2yr x 4yr swaption
* the 1yr x 2yr swaption

Report these values and compare them to the price of the `1y x 4y` swaption.


In [10]:
atm_implied_vol_1y_4y = sw_1y_4y_vols[0].values[0]
alternate_swaptions = [(0.25, 4), (2, 4), (1, 2)]
alt_prices = []
for expiry, tenor in alternate_swaptions:
    fw_sw_rate_df = calc_forward_swap_rate(forward_rates_df, expiry)
    # FIX: use a local variable so the outer fw_sw_rate (1x4 ATM rate) is NOT overwritten
    fw_sw_rate_alt = fw_sw_rate_df.loc[fw_sw_rate_df['tenor'] == expiry + tenor, 'forward_swap_rate'].values[0]
    vol = atm_implied_vol_1y_4y
    print(f'atm swap rate for expiry {expiry} and tenor {tenor}: {fw_sw_rate_alt}')
    # FIX: pass forward_rates_df (full dataset) so the tenor-filter inside
    #      calc_swaption_price works correctly for all expiry/tenor combinations
    price = calc_swaption_price(fw_sw_rate_alt, fw_sw_rate_alt, vol,
                                forward_rates_df, fraction=fraction,
                                expiry=expiry, tenor=tenor) * notional
    alt_prices.append(price)
    print(f'Swaption with expiry {expiry} and tenor {tenor} has price: {price}')

display(pd.DataFrame({
    'expiry': [x[0] for x in alternate_swaptions],
    'tenor':  [x[1] for x in alternate_swaptions],
    'price':  alt_prices
}).style.format({'price': '{:,.2f}'}))
print(f'for expiry 1 and tenor 4 (original), the price is {atm_price:,.2f}')

atm swap rate for expiry 0.25 and tenor 4: 0.0329975899972315
discount_factor_sw: 3.692193591655822, d1: 0.07707499999999999, d2: -0.07707499999999999
Swaption with expiry 0.25 and tenor 4 has price: 0.7484976817723415
atm swap rate for expiry 2 and tenor 4: 0.03425670508218368
discount_factor_sw: 3.4844951401318007, d1: 0.21800102063981258, d2: -0.21800102063981258
Swaption with expiry 2 and tenor 4 has price: 2.0599419813611464
atm swap rate for expiry 1 and tenor 2: 0.031183436805103607
discount_factor_sw: 1.8605044915911473, d1: 0.15414999999999998, d2: -0.15414999999999998
Swaption with expiry 1 and tenor 2 has price: 0.7107568388902671


,expiry,tenor,price
0,0.250000,4,0.75
1,2.000000,4,2.06
2,1.000000,2,0.71


for expiry 1 and tenor 4 (original), the price is 1.44


- The 1yr expiry 4yr tenor swaption's price is in between the 1yr expiry 2yr tenor swaption's price of 0.71 and 2yr expiry 4yr tenor swaption's price of 2.06. 
- We can see that with increase in the expiry of the option and with the same tenor, there is an increase in price, which is very similar to equitities i.e., options with higher expiry have higher price becuase the underlying has a higher amount of time to go ITM.